# Importations

In [1]:
# Numerical and scientific python programming
import numpy as np

from scipy.stats import special_ortho_group
from scipy.linalg import eigh

# Auxiliary python functions
from itertools import combinations, product
from functools import reduce
from typing import Sequence, Iterable

# Local importations
from moments.bloch import (compute_pauli_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bloch_vector, compute_dm_from_bloch, compute_bloch_norms_from_vector)
from moments.quantum import generate_rand_dm, compute_is_valid_dm


# Definitions

---

**Observation 1.** Any fully separable three-qubit state obeys

$$
3 S_3 + S_2 \le 3 + S_1
$$

or, equivalently,

$$
3 || \vec r_{123} ||^2 + || \vec r_{12} ||^2 + || \vec r_{12} ||^2 + || \vec r_{23} ||^2 \le 3 + || \vec r_1 ||^2 + || \vec r_2 ||^2 + || \vec r_3 ||^2 \, .
$$

This is the optimal linear criterion in the sense that any other linear criterion for the Ai detects strictly fewer states.

---

---

**Observation 2.** Any three-qubit state which is separable with respect to some bipartition obeys

$$
S_3 + S_2 ≤ 3(1 + S_1)
$$

or, equivalently,

$$
|| \vec r_{123} ||^2 + || \vec r_{12} ||^2 + || \vec r_{12} ||^2 + || \vec r_{23} ||^2 \le 3(1 + || \vec r_1 ||^2 + || \vec r_2 ||^2 + || \vec r_3 ||^2) \, .
$$

This is the optimal criterion in the sense that if the three $S_k$ obey the inequality, then for any bipartition there is a separable state compatible with them.

---

Whithin the purity constraints, Bloch lengths and Sector lengths satisfy:

- $||\vec r_k||^2 \in [0, 1]$, then $S_1 \in [0, 3]$,
- $||\vec r_{kl}||^2 \in [0, 3]$, then $S_2 \in [0, 9]$,
- $||\vec r_{123}||^2 = S_3 \in [0, 7]$;

with $k, l = 1, 2, 3$. These bounds are not tight.

In [2]:
def compute_rand_SO_subset(subset_index_map):
    Q = {}
    for subset in subset_index_map.keys():
        Q[subset] = special_ortho_group.rvs(len(subset_index_map[subset]))
    return Q

def check_tensor_rot(Q):
    for subset in Q.keys():
        if len(subset) > 1:
            Q_tensor = np.array([1])
            for m in subset:
                Q_tensor = np.kron(Q_tensor, Q[(m,)])
            if np.allclose(Q[subset], Q_tensor):
                return False
    return True

def tensor_product(O_v: Sequence[np.ndarray]) -> np.ndarray:
    try:
        return reduce(np.kron, O_v)
    except ValueError as e:
        raise ValueError("Some arrays in O_v have incompatible shapes for Kronecker product.") from e

def compute_pt(dim: list[int], A: np.ndarray, subsystem: int | Iterable[int] = 1,) -> np.ndarray:
    """
    Compute the partial transpose of a multipartite operator.

    Parameters
    ----------
    dim : list[int]
        Local Hilbert-space dimensions.
    A : ndarray
        Matrix of shape (prod(dim), prod(dim)).

    subsystem : int or iterable[int]
        Which subsystem(s) to transpose.

    Returns
    -------
    ndarray
        Partial transpose.
    """

    nsub = len(dim)
    D = np.prod(dim)

    if A.shape != (D, D):
        raise ValueError("Matrix shape inconsistent with dimensions.")

    if isinstance(subsystem, int):
        subsystem = [subsystem]
    subsystem = set(subsystem)

    tensor = A.reshape(*dim, *dim)

    perm = list(range(2 * nsub))

    for s in subsystem:
        perm[s], perm[s + nsub] = perm[s + nsub], perm[s]

    tensor_pt = tensor.transpose(perm)

    return tensor_pt.reshape(D, D)

def compute_tr_norm(eigenvalues: np.ndarray | None = None, A: np.ndarray | None = None) -> float:
    """
    Compute the trace norm of a Hermitian matrix.
    For a Hermitian matrix A, the trace norm reduces to the sum of absolute eigenvalues.

    Parameters
    ----------
    eigenvalues : np.ndarray, optional
        Precomputed eigenvalues of A. If provided, A is not used.
    A : np.ndarray, optional
        A Hermitian matrix of shape (d, d). Required if eigenvalues is not provided.
    
    Returns
    -------
    float
        The trace norm of A.
    """
    if eigenvalues is None:
        if A is None:
            raise ValueError("Provide eigenvalues or matrix.")
        if A.shape[0] != A.shape[1]:
            raise ValueError("Matrix must be square.")
        if not np.allclose(A, A.conj().T):
            raise ValueError("Matrix must be Hermitian.")
        eigvals = eigh(A, eigvals_only=True)
    
    return float(np.sum(np.abs(eigvals)))

def compute_negativity(rho: np.ndarray, dim: list[int], subsystem: int | Iterable[int] = 1) -> float:
    """
    Compute the entanglement negativity of a quantum state by specifying the bipartition.

    Parameters
    ----------
    rho : np.ndarray
        Density matrix of shape (dA*dB, dA*dB)
    dim : List[int]
        Local Hilbert space dimensions.
    subsystem : int
        Subsystem to partially transpose.

    Returns
    -------
    float
        The entanglement negativity.
    """
    rho_pt = compute_pt(dim, rho, subsystem)
    return (compute_tr_norm(A=rho_pt) - 1.) / 2.

def compute_tripartite_negativity(rho: np.ndarray, dim: list[int]) -> float:
    """
    Compute the tripartite negativity.

    Parameters
    ----------
    rho : ndarray
        Three-party density matrix.

    dim : list[int]
        Local dimensions, e.g. [2,2,2].

    Returns
    -------
    float
        The tripartite negativity.
    """
    if len(dim) != 3:
        raise ValueError("Tripartite negativity requires three subsystems.")

    negativities = [
        compute_negativity(rho=rho, dim=dim, subsystem=i)
        for i in range(3)
    ]

    return float(np.prod(negativities) ** (1 / 3))

In [3]:
dn, N = 2, 3
dim = [dn]*N
d = int(np.prod(dim))

pauli_basis = compute_pauli_basis()

local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Viability test

In [4]:
is_valid = False
j = 0

while not is_valid:
    j += 1
    
    rho = generate_rand_dm(d, d)
    r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

    Q = compute_rand_SO_subset(subset_index_map)
    if not check_tensor_rot(Q):
        print("Warning: Q_M is equal to the tensor product of Q_m for some M")
    
    r_rot = {subset: Q[subset] @ r[subset] for subset in subset_index_map.keys()}
    
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    is_valid, _ = compute_is_valid_dm(rho_rot)

print("Sucess:", is_valid)
print("Number of iterations:", j)


Sucess: True
Number of iterations: 1931


# Local identity rotations

$$
Q_n = \mathbb I_3 \, , \quad \forall n \in N
$$
$$
Q_{nm} = \mathbb I_9 \, , \quad \forall n, m \in N
$$

In [5]:
is_valid = False
j = 0

while not is_valid:
    j += 1

    rho = generate_rand_dm(d, d)
    r = compute_bloch_vector(tensor_basis, subset_index_map, rho)

    Q_123 = special_ortho_group.rvs(27)
    if np.allclose(Q_123, np.identity(27)):
        print("Warning: Q_123 is equal to the tensor product of Q_1, Q_2 and Q_3n(i.e. the identity).")
    
    r_rot = r.copy()
    r_rot[(1, 2, 3)] = Q_123 @ r_rot[(1, 2, 3)]
    
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    
    is_valid, _ = compute_is_valid_dm(rho_rot)

print("Sucess:", is_valid)
print("Number of iterations:", j)

Sucess: True
Number of iterations: 3682


In [6]:
R = compute_bloch_norms_from_vector(r)
R_rot = compute_bloch_norms_from_vector(r_rot)

bloch_diff = {subset: np.allclose(r[subset], r_rot[subset]) for subset in subset_index_map}
length_diff = {subset: np.allclose(R[subset], R_rot[subset]) for subset in subset_index_map}
DN = abs(compute_tripartite_negativity(rho, dim) - compute_tripartite_negativity(rho_rot, dim))

print("One-body Bloch vectors are constant:", (bloch_diff[(1,)] and bloch_diff[(2,)]))
print("Two-body Bloch vector is constant:", bloch_diff[(1, 2)])
print("Three-body Bloch vector is constant:", bloch_diff[(1, 2, 3)])
print("Density matrix is constant:", np.allclose(rho, rho_rot))
print("Bloch lengths are constant:", all(list(length_diff.values())))
print(f"Tripartite negativity difference: {DN:.4f}")

One-body Bloch vectors are constant: True
Two-body Bloch vector is constant: True
Three-body Bloch vector is constant: False
Density matrix is constant: False
Bloch lengths are constant: True
Tripartite negativity difference: 0.0037


# Exact rotations

In [7]:
def iterate_k_nonzero_arrays(k, size, min_value = 1, max_value = 10):
    
    base_array = np.zeros(size)
    
    value_range = range(min_value, max_value + 1)
    
    for indices in combinations(range(size), k):
        
        for values in product(value_range, repeat = k):
            
            array = base_array.copy()
            array[list(indices)] = values
            
            yield array

In [8]:
z = np.array([0, 0, 1])
zz = tensor_product([z, z])
zzz = tensor_product([z, z, z])
a, b = 0.5, 0.5
r = {(1,): a * z, (2,): a * z, (3,): a * z,
     (1, 2): b * zz, (1, 3): b * zz, (2, 3): b * zz,
     (1, 2, 3): zzz}

rho = compute_dm_from_bloch(tensor_basis, subset_index_map, r)
is_valid, _ = compute_is_valid_dm(rho)

N = compute_tripartite_negativity(rho, dim)

print("Original state:\n")
print(rho)
print("\nIs a valid density matrix?", is_valid)
print("Tripartite negativity:", N)

Original state:

[[0.625+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.125+0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.125+0.j 0.   +0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.125+0.j
  0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
  0.   +0.j]]

Is a valid density matrix? True
Tripartite negativity: 0.0


In [9]:
k = 4
results = []
negativities = []

for j, r_123 in enumerate(iterate_k_nonzero_arrays(k, size = 27, max_value = 2)):
    
    r_rot = r.copy()
    r_rot[(1, 2, 3)] = r_123 / np.linalg.norm(r_123)
    rho_rot = compute_dm_from_bloch(tensor_basis, subset_index_map, r_rot)
    
    is_valid, _ = compute_is_valid_dm(rho_rot)
    if is_valid:
        print(f'\n# {j}:\nr_12 = {r_123}.')
        N_rot = compute_tripartite_negativity(rho_rot, dim)
        print(f"N = {N_rot:.4f}")
        results.append(j)
        negativities.append(N_rot)


# 47872:
r_12 = [0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1.].
N = 0.1250

# 47887:
r_12 = [0. 2. 0. 2. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 2.].
N = 0.1250

# 153328:
r_12 = [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1.].
N = 0.1250

# 153343:
r_12 = [0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 2.].
N = 0.1250


In [10]:
j_max = np.argmax(np.array(negativities))

print("Maximum concurrence")
print(f"N = {negativities[j_max]:.4f}")

Maximum concurrence
N = 0.1250
